# Macaron food photo model

This notebook trains the small model that lets the Macaron app recognise a dish from a photo, on the phone, with no internet.

**What it does**

1. Downloads two public food photo datasets.
2. Fine-tunes a small pretrained network (MobileNetV3) to tell the dishes apart.
3. Saves two files for the app: `food.onnx` (the model) and `labels.json` (the dish names).

**How to run it on Kaggle**

1. Go to kaggle.com, sign in, and choose **Create** then **New Notebook**.
2. **File** then **Import Notebook**, and upload this file.
3. In the right-hand panel, open **Session options**. Set **Accelerator** to **GPU T4 x2** and **Internet** to **On**.
4. Press **Run All** at the top. It takes roughly 30 to 60 minutes.
5. When it finishes, open the **Output** section on the right. Download `food.onnx` and `labels.json`.
6. Put both files in the `model` folder of the app's repository.

The model can only recognise the dishes listed in `MAP` below. Each entry links a dataset class to the exact dish name used in the app.

In [ ]:
!pip -q install timm datasets onnx onnxruntime
!pip -q install ddgs


In [ ]:
import json, random, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from datasets import load_dataset
from PIL import Image

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", DEVICE)

# dataset class -> exact dish name in the app's food list
MAP_MY = {  # lowyisan/malaysian_food_images
    "fish_and_chips": "Fish and chips", "fried_noodles": "Mee goreng mamak", "fried_rice": "Nasi goreng kampung",
    "hamburger": "Cheeseburger", "kaya_toast": "Kaya toast set", "laksa": "Curry laksa",
    "mixed_rice": "Economy rice: 1 meat, 2 veg", "nasi_lemak": "Nasi lemak biasa", "popiah": "Popiah, fresh",
    "roti_canai": "Roti canai kosong", "satay": "Satay ayam",
}
MAP_F101 = {  # ethz/food101, only the classes that are common in Malaysia and exist in the app
    "sushi": "Sushi plates", "sashimi": "Salmon sashimi and rice", "ramen": "Ramen, shoyu", "bibimbap": "Bibimbap",
    "pad_thai": "Pad thai", "pizza": "Pizza", "spaghetti_bolognese": "Spaghetti bolognese",
    "spaghetti_carbonara": "Spaghetti carbonara", "caesar_salad": "Chicken caesar salad", "french_fries": "French fries",
    "ice_cream": "Ice cream", "chocolate_cake": "Chocolate cake", "cheesecake": "Cheesecake", "donuts": "Doughnut",
    "hummus": "Hummus and falafel plate", "falafel": "Hummus and falafel plate", "edamame": "Edamame",
    "miso_soup": "Miso soup", "chicken_wings": "Fried chicken", "dumplings": "Har gow",
    "omelette": "Omelette with rice and cucumber", "grilled_salmon": "Pan-seared salmon, rice and greens",
    "gyoza": "Gyoza", "takoyaki": "Takoyaki", "pho": "Pho", "spring_rolls": "Spring rolls, fried",
    "chicken_curry": "Chicken curry with rice", "peking_duck": "Roast duck rice", "waffles": "Waffle with syrup",
    "pancakes": "Pancakes with syrup", "french_toast": "French toast", "club_sandwich": "Club sandwich",
    "steak": "Steak with vegetables", "hot_dog": "Hot dog", "tiramisu": "Tiramisu", "lasagna": "Lasagna",
    "macaroni_and_cheese": "Macaroni and cheese", "garlic_bread": "Garlic bread", "samosa": "Samosa",
    "cup_cakes": "Cupcake", "frozen_yogurt": "Frozen yogurt", "tacos": "Tacos", "macarons": "Macarons",
    "fried_calamari": "Fried calamari", "greek_salad": "Garden salad", "eggs_benedict": "Eggs Benedict",
    "apple_pie": "Apple pie",
}
MAP_IN = {  # rajistics/indian_food_images (small, so it only tops up these classes)
    "chapati": "Chapati with dhal", "idli": "Idli with sambar", "masala_dosa": "Masala thosai",
    "chai": "Teh tarik", "samosa": "Samosa",
}
PER_CLASS = 500      # cap on training photos per class, to keep classes balanced
EPOCHS_HEAD, EPOCHS_ALL = 2, 8
BATCH = 64

## Load the photos

Both datasets come from the Hugging Face Hub. Food-101 is large (about 5 GB), so this step takes a few minutes.

In [ ]:
my = load_dataset("lowyisan/malaysian_food_images")
f101 = load_dataset("ethz/food101")

my_names = my["train"].features["label"].names
f_names = f101["train"].features["label"].names

try:
    india = load_dataset("rajistics/indian_food_images")
    in_names = india["train"].features["label"].names
except Exception as e:                                  # optional extra data: carry on without it
    india = None; MAP_IN = {}; print("Skipping the Indian food dataset:", e)


# ---- extra Malaysian hawker and home dishes from Wikimedia Commons ----
# Photos on Wikimedia Commons are published under licences that allow reuse, and the site offers an API for this.
# Search results are noisy (not every photo shows the dish), so expect lower accuracy for these classes.
import os, re, io, time, requests
COMMONS = {
 "ayam masak kicap": "Ayam masak kicap with rice and sayur",
 "kari ayam": "Chicken curry with rice",
 "steamed fish chinese": "Steamed fish with rice and greens",
 "kangkung belacan stir fried": "Stir-fried greens",
 "dal curry": "Dhal curry",
 "roast chicken potatoes home": "Baked chicken thigh with potatoes and salad",
 "braised tofu": "Tofu, braised",
 "tempeh goreng": "Tempeh, fried",
 "char kway teow": "Char kway teow",
 "bak kut teh": "Bak kut teh with rice",
 "hainanese chicken rice": "Chicken rice, steamed",
 "nasi briyani": "Nasi briyani ayam",
 "dosa": "Thosai with dhal and chutney",
 "chapati": "Chapati with dhal",
 "idli": "Idli with sambar",
 "wonton noodles": "Wantan mee, dry",
 "asam laksa": "Asam laksa",
 "hokkien mee kuala lumpur": "Hokkien mee (KL)",
 "mee rebus": "Mee rebus",
 "lontong": "Lontong",
 "nasi kandar": "Nasi kandar",
 "murtabak": "Murtabak ayam",
 "cendol": "Cendol",
 "ais kacang": "Ais kacang (ABC)",
 "apam balik": "Apam balik",
 "pisang goreng": "Pisang goreng",
 "curry puff": "Curry puff",
 "teh tarik": "Teh tarik",
 "rojak buah": "Rojak buah",
 "yong tau foo": "Yong tau foo, soup",
 "claypot chicken rice": "Claypot chicken rice",
 "chee cheong fun": "Chee cheong fun, plain",
 "prawn mee": "Prawn mee",
 "pan mee": "Pan mee, soup",
 "nasi dagang": "Nasi dagang",
 "soto ayam": "Soto ayam",
 "ikan bakar": "Ikan bakar with rice and ulam",
 "tandoori chicken": "Tandoori chicken with naan",
 "kuih": "Kuih, 2 pieces",
 "roti telur": "Roti telur",
 "mee siam": "Mee siam",
 "dim sum": "Dim sum breakfast",
 "char siu bao": "Char siu bao",
 "siu mai": "Siew mai",
 "lo mai gai": "Lor mai kai",
 "ayam penyet": "Nasi ayam penyet",
 "fried egg": "Fried egg",
 "boiled egg": "Boiled egg",
 "steamed white rice bowl": "White rice",
 "douhua tofu pudding": "Tau fu fah, less syrup",
 "keropok lekor": "Keropok lekor",
 "sup ayam": "Sup ayam with rice",
 "nasi lemak": "Nasi lemak biasa",
 "roti canai": "Roti canai kosong",
 "satay": "Satay ayam",
 # drinks, desserts, snacks, dim sum and pastries
 "yakult bottle": "Yakult Ace",
 "vitagen cultured milk": "Vitagen",
 "brown sugar boba milk": "Brown sugar pearl fresh milk",
 "bubble tea": "Bubble milk tea",
 "taro milk tea": "Taro milk tea with pearls",
 "iced matcha latte": "Matcha latte, iced",
 "chocolate milkshake": "Chocolate milkshake",
 "mango lassi": "Mango lassi",
 "mango smoothie": "Mango Magic smoothie",
 "berry smoothie": "All Berry Bang smoothie",
 "sugarcane juice": "Sugarcane juice",
 "kopi ais iced coffee": "Kopi ais",
 "teh ais iced milk tea": "Teh ais",
 "caramel frappuccino": "Caramel frappé",
 "iced chocolate drink": "Iced chocolate",
 "milo ais": "Milo ais",
 "watermelon juice": "Watermelon juice, no sugar",
 "caffe latte": "Latte",
 "americano coffee": "Americano",
 "cola can": "Soft drink",
 "frozen yogurt toppings": "llaollao, medium tub with 3 toppings",
 "ice cream scoop cup": "Inside Scoop, single scoop",
 "ice cream cone scoops": "Inside Scoop, double scoop in a waffle cone",
 "waffle with ice cream": "Ice cream waffle",
 "souffle pancake": "Soufflé pancakes",
 "mango bingsu": "Mango bingsu",
 "injeolmi bingsu": "Injeolmi bingsu",
 "chocolate bingsu": "Chocolate bingsu",
 "mango pomelo sago": "Mango sago pomelo",
 "red bean soup dessert": "Red bean soup",
 "bubur cha cha": "Bubur cha cha",
 "mango sticky rice": "Mango sticky rice",
 "daifuku mochi": "Mochi",
 "chocolate brownie": "Brownie",
 "chocolate chip cookie": "Chocolate chip cookie",
 "churros": "Churros with chocolate dip",
 "banana chocolate crepe": "Crêpe with banana and chocolate",
 "molten chocolate cake": "Molten lava cake with ice cream",
 "mcflurry": "McFlurry with Oreo",
 "chocolate sundae": "Chocolate sundae",
 "tiramisu": "Tiramisu",
 "cheesecake slice": "Cheesecake",
 "chocolate cake slice": "Chocolate cake",
 "macarons": "Macarons",
 "glazed doughnut": "Doughnut",
 "popcorn": "Popcorn, salted",
 "potato chips": "Potato chips",
 "chocolate bar": "Chocolate bar",
 "granola bar": "Granola bar",
 "soft serve ice cream": "Soft serve cone",
 "xiaolongbao": "Xiao long bao",
 "har gow": "Har gow",
 "steamed spare ribs dim sum": "Steamed pork ribs with black bean",
 "chicken feet dim sum": "Chicken feet in black bean sauce",
 "shrimp rice noodle roll": "Prawn cheong fun",
 "turnip cake dim sum": "Lo bak go, pan-fried radish cake",
 "taro dumpling wu gok": "Yam puff (wu kok)",
 "fried wonton": "Fried wantan",
 "shengjian mantou": "Sheng jian bao",
 "liu sha bao custard bun": "Salted egg custard bun (lau sar bao)",
 "ma lai go sponge cake": "Ma lai gao",
 "jian dui sesame ball": "Sesame balls (jin deui)",
 "char siu sou pastry": "Char siu sou",
 "mango pudding": "Mango pudding",
 "century egg congee": "Century egg and pork porridge",
 "zongzi": "Bak chang",
 "egg tart": "Egg tart",
 "fried spring rolls": "Spring rolls, fried",
 "gyoza": "Gyoza",
 "fried mantou condensed milk": "Fried mantou with condensed milk",
 "croissant": "Butter croissant",
 "almond croissant": "Almond croissant",
 "pain au chocolat": "Pain au chocolat",
 "danish pastry": "Fruit Danish",
 "cinnamon roll": "Cinnamon roll",
 "blueberry muffin": "Blueberry muffin",
 "chocolate muffin": "Chocolate muffin",
 "banana bread": "Banana bread",
 "scone jam cream": "Scone with jam and cream",
 "cream puff": "Cream puff",
 "chocolate eclair": "Chocolate éclair",
 "swiss roll cake": "Swiss roll",
 "pandan chiffon cake": "Pandan chiffon cake",
 "red velvet cake": "Red velvet cake",
 "carrot cake slice": "Carrot cake with cream cheese",
 "basque burnt cheesecake": "Burnt cheesecake",
 "mille crepe cake": "Mille crêpe",
 "fruit tart": "Fruit tart",
 "lemon tart": "Lemon tart",
 "pastel de nata": "Portuguese egg tart",
 "pineapple tart": "Pineapple tarts",
 "mooncake": "Mooncake, lotus paste",
 "taiyaki": "Taiyaki",
 "sausage roll": "Chicken sausage roll",
 "chicken pie": "Chicken pie",
 "quiche": "Quiche",
 "soft pretzel": "Pretzel, original",
 "bagel cream cheese": "Bagel with cream cheese",
 "pineapple bun bolo bao": "Polo bun with butter",
 "coffee bun rotiboy": "Mexican coffee bun",
 "sliced white bread": "White bread",
 "cupcake": "Cupcake",
 "apple pie": "Apple pie",
 "waffle syrup": "Waffle with syrup",
 "pancakes syrup": "Pancakes with syrup",
 # tong sui, bagels and trending treats
 "green bean soup dessert": "Green bean soup",
 "black sesame soup": "Black sesame paste",
 "peanut soup dessert chinese": "Peanut paste",
 "walnut soup dessert chinese": "Walnut paste",
 "almond tea dessert chinese": "Almond paste",
 "sweet potato soup ginger dessert": "Sweet potato ginger soup",
 "tangyuan": "Tang yuan in ginger syrup",
 "snow fungus soup": "Snow fungus with longan",
 "barley ginkgo dessert soup": "Barley, fu chuk and gingko",
 "bubur pulut hitam": "Bubur pulut hitam",
 "honeydew sago": "Honeydew sago",
 "leng chee kang": "Leng chee kang",
 "guilinggao": "Gui ling gao herbal jelly",
 "double skin milk": "Double-skin milk pudding",
 "taro sago dessert": "Taro sago",
 "bagel": "Bagel, plain",
 "lox bagel": "Smoked salmon and cream cheese bagel",
 "egg cheese bagel sandwich": "Egg and cheese bagel",
 "blueberry bagel": "Blueberry bagel with cream cheese",
 "shio pan salt bread": "Salt bread (shio pan)",
 "korean cream cheese garlic bread": "Korean garlic cream cheese bun",
 "dubai chocolate": "Dubai chocolate bar",
 "dubai chewy cookie": "Dubai chewy cookie",
 "dubai chocolate strawberries cup": "Dubai chocolate strawberry cup",
 "kunafa": "Kunafa",
 "cromboloni": "Cromboloni",
 "crookie croissant cookie": "Crookie",
 "croffle": "Croffle",
 "mochi donut pon de ring": "Mochi doughnut",
 "matcha tiramisu": "Matcha tiramisu",
 "banana pudding": "Banana pudding",
 "tanghulu": "Tanghulu",
 "corn cheese cup street food": "Jagung cheese",
 "fruit sando": "Fruit sando",
 "durian flesh": "Musang King durian",
 "durian cendol": "Durian cendol",
 "pandan gula melaka cake": "Pandan gula melaka cake",
 "caramel cheese popcorn": "Garrett popcorn, Chicago mix",
 "soft serve cone": "Mixue ice cream cone",
 "strawberry matcha latte": "Strawberry matcha latte",
 "pistachio latte": "Pistachio latte",
 "einspanner coffee": "Einspänner cream coffee",
 "coconut shake": "Coconut shake",
 # home-style dishes
 "fish soup tofu tomato": "Fish slice soup with tofu and tomato",
 "minced pork soup": "Minced pork soup",
 "fried fish cake": "Fried fish cake",
 "quesadilla": "Chicken quesadilla",
 # Korean BBQ and the newer dishes
 "samgyeopsal korean bbq grill": "Korean BBQ, typical meal",
 "bulgogi": "Bulgogi with rice",
 "kimchi jjigae": "Kimchi jjigae with rice",
 "sundubu jjigae": "Sundubu jjigae with rice",
 "tteokbokki": "Tteokbokki",
 "jajangmyeon": "Jajangmyeon",
 "samgyetang": "Samgyetang",
 "budae jjigae": "Army stew (budae jjigae)",
 "pajeon": "Seafood pajeon",
 "tonkatsu": "Tonkatsu set",
 "unadon": "Unagi don",
 "karaage": "Chicken karaage",
 "okonomiyaki": "Okonomiyaki",
 "yakitori": "Yakitori",
 "california roll": "California roll",
 "zaru soba": "Cold soba",
 "thai green curry": "Green curry chicken with rice",
 "som tam": "Som tam",
 "thai iced tea": "Thai milk tea",
 "char siu rice": "Char siu rice",
 "siu yuk": "Siew yuk rice",
 "sweet and sour pork": "Sweet and sour pork with rice",
 "kung pao chicken": "Kung pao chicken with rice",
 "kolo mee": "Kolo mee",
 "sarawak laksa": "Sarawak laksa",
 "nasi kerabu": "Nasi kerabu with ayam percik",
 "rendang": "Rendang daging with rice",
 "butter chicken": "Butter chicken with naan",
 "banana leaf rice": "Banana leaf rice with chicken",
 "vada": "Vadai",
 "barbecue chicken wings": "BBQ chicken wings",
 "barbecue pork ribs": "BBQ pork ribs",
 "lamb chops plate": "Lamb chop with fries",
 "full english breakfast": "Big breakfast",
 "avocado toast egg": "Avocado toast with egg",
 "big mac": "Big Mac",
 "chinese hot pot": "Hot pot, typical meal",
 "otak-otak": "Otak-otak"
}
UA = {"User-Agent": "MacaronFoodModel/1.1 (personal, non-commercial food classifier; https://github.com/SiobhanGoh/macaron)"}

def api(params, tries=6):
    """Wikimedia asks bots to go slowly. Retry with a growing pause when it says no."""
    for k in range(tries):
        try:
            r = requests.get("https://commons.wikimedia.org/w/api.php", headers=UA, timeout=30, params=params)
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/json"):
                return r.json()
        except Exception:
            pass
        time.sleep(3 * 2 ** k)
    raise RuntimeError("Wikimedia Commons kept refusing")

def commons(term, want=120):
    out, cont = [], {}
    while len(out) < want:
        r = api({"action": "query", "format": "json", "generator": "search", "gsrsearch": f"filetype:bitmap {term}",
                 "gsrnamespace": 6, "gsrlimit": 50, "prop": "imageinfo", "iiprop": "url|mime", "iiurlwidth": 512,
                 "maxlag": 5, **cont})
        for pg in r.get("query", {}).get("pages", {}).values():
            ii = (pg.get("imageinfo") or [{}])[0]
            if ii.get("mime") in ("image/jpeg", "image/png") and ii.get("thumburl"): out.append(ii["thumburl"])
        if "continue" not in r: break
        cont = r["continue"]; time.sleep(1.0)
    return out[:want]

# ---- optional: top up each dish with photos from a public image search ----
# These are other people's photos. They are used here only to teach the model and are never shown in the app
# or shared. Keep it that way: do not publish the downloaded folder. Set WEB_PHOTOS = False to skip this.
WEB_PHOTOS = True
WEB_PER_DISH = 100

def web_photos(term, want):
    q = term + " food"
    try:
        try: from ddgs import DDGS
        except ImportError: from duckduckgo_search import DDGS
        for k in range(3):
            try: return [r["image"] for r in DDGS().images(q, max_results=want) if r.get("image")]
            except Exception as e: err = e; time.sleep(6 * (k + 1))
        raise err
    except Exception as e:
        print("  image search unavailable for", term, "-", str(e)[:80]); return []

from concurrent.futures import ThreadPoolExecutor
def grab(job):
    u, fn = job
    try:
        if not os.path.exists(fn):
            im = Image.open(io.BytesIO(requests.get(u, headers=UA, timeout=20).content)).convert("RGB")
            im.thumbnail((512, 512)); im.save(fn, quality=88)
        return fn
    except Exception: return None

slug = lambda t: "cm_" + re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")
MAP_CM, CM_FILES = {}, {}
for term, food in COMMONS.items():
    lab = slug(term); d = f"/tmp/commons/{lab}"; os.makedirs(d, exist_ok=True); files = []   # /tmp is kept out of the saved outputs, so no photo is ever published
    try: urls = commons(term)
    except Exception as e: print("skipped", term, e); continue
    jobs = [(u, f"{d}/{i}.jpg") for i, u in enumerate(urls)]
    if WEB_PHOTOS: jobs += [(u, f"{d}/w{j}.jpg") for j, u in enumerate(web_photos(term, WEB_PER_DISH))]; time.sleep(1.0)
    time.sleep(2.0)
    with ThreadPoolExecutor(4) as pool: files = [f for f in pool.map(grab, jobs) if f]
    if len(files) >= 25: MAP_CM[lab] = food; CM_FILES[lab] = files
    print(f"{term}: {len(files)} photos" + ("" if len(files) >= 25 else "  (too few, left out)"))

# ---- dishes from the restaurants in the big malls (1 Utama, Sunway Pyramid, KL) ----
# Many shop dishes look the same in a photo (every shop's xiao long bao looks alike), so each shop dish
# teaches the shared "look" dish. The app then lists the shop versions after a scan.
# Photos come from the same public image search as above, stay in /tmp and are never published.
SHOP_LOOKS = {
"Absolute Thai pad thai with prawns": "Pad thai",
"Absolute Thai green curry chicken with rice": "Green curry chicken with rice",
"Absolute Thai tom yum goong": "Tom yum soup with rice",
"Absolute Thai basil chicken rice with fried egg": "Chicken rice, steamed",
"Absolute Thai pineapple fried rice": "Nasi goreng kampung",
"Absolute Thai mango sticky rice": "Mango sticky rice",
"Thai milk tea, iced": "Bubble milk tea",
"AK Noodles pan mee, dry": "Pan mee, soup",
"AK Noodles pan mee, soup": "Pan mee, soup",
"AK Noodles chili pan mee": "Pan mee, soup",
"AK Noodles wantan mee": "Wantan mee, dry",
"ANTE kurobuta pork chop": "Hainanese chicken chop",
"ANTE pork belly aglio olio": "Spaghetti carbonara",
"ANTE carbonara with bacon": "Spaghetti carbonara",
"ANTE pork burger": "Cheeseburger",
"Ayam pongteh with rice": "Green curry chicken with rice",
"Nyonya laksa": "Curry laksa",
"Nyonya curry kapitan chicken with rice": "Green curry chicken with rice",
"Nyonya chap chye with rice": "Stir-fried greens",
"Pai tee": "Kuih, 2 pieces",
"Cendol, Nyonya style": "Cendol",
"BananaBro nasi lemak ayam goreng berempah": "Nasi lemak biasa",
"BananaBro nasi lemak rendang daging": "Nasi lemak biasa",
"BananaBro nasi lemak sotong sambal": "Nasi lemak biasa",
"BananaBro nasi lemak biasa": "Nasi lemak biasa",
"Banglo 289 nasi goreng kampung": "Nasi goreng kampung",
"Banglo 289 ayam penyet with rice": "Nasi ayam penyet",
"Banglo 289 mee goreng mamak": "Mee goreng mamak",
"Bar B Q Plaza pork set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza chicken set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza seafood set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza jasmine rice": "White rice",
"Bask Bear iced latte": "Latte, iced",
"Bask Bear Spanish latte, iced": "Latte, iced",
"Bask Bear Coconut latte, iced": "Latte, iced",
"Bask Bear Hazelnut latte, iced": "Latte, iced",
"Bask Bear matcha latte, iced": "Matcha latte, iced",
"bhc Bburinkle chicken": "Korean fried chicken, soy garlic",
"bhc Matchoking chicken": "Korean fried chicken, soy garlic",
"bhc Hot Fried chicken": "Korean fried chicken, soy garlic",
"bhc cheese balls": "Chicken nuggets",
"Black Canyon tom yum fried rice": "Nasi goreng kampung",
"Black Canyon pad kra pao chicken rice": "Chicken rice, steamed",
"Black Canyon green curry with rice": "Green curry chicken with rice",
"Black Canyon iced cappuccino": "Chocolate milkshake",
"BMS Organics brown rice nasi lemak": "Nasi lemak biasa",
"BMS Organics vegetarian laksa": "Curry laksa",
"BMS Organics quinoa salad": "Chicken caesar salad",
"BMS Organics alkaline juice": "Soft drink",
"Bungkus Kaw Kaw nasi lemak ayam": "Nasi lemak biasa",
"Bungkus Kaw Kaw nasi lemak sotong": "Nasi lemak biasa",
"Bungkus Kaw Kaw teh tarik": "Teh tarik",
"Canton-i wantan noodles": "Wantan mee, dry",
"Canton-i century egg and pork congee": "Chicken porridge",
"Canton-i fish slice congee": "Chicken porridge",
"Canton-i beef brisket noodles": "Pho",
"Canton-i Hong Kong milk tea": "Bubble milk tea",
"Canton-i char siew rice": "Char siu rice",
"Champignons croissant": "Butter croissant",
"Champignons almond croissant": "Butter croissant",
"Champignons fruit tart": "Fruit tart",
"Champignons chocolate cake": "Chocolate cake",
"Chili's Oldtimer with cheese burger": "Cheeseburger",
"Chili's Southwestern eggrolls": "Pisang goreng",
"Chili's chicken fajitas": "Bagel with cream cheese",
"Chili's baby back ribs, half rack": "BBQ pork ribs",
"Chili's chicken crispers": "Chicken nuggets",
"Chili's Molten Chocolate Cake": "Chocolate cake",
"Chili's Presidente margarita, non-alcoholic": "Soft drink",
"Vegetarian economy rice, 3 dishes": "Economy rice: 1 meat, 2 veg",
"Vegetarian curry mee": "Curry laksa",
"Vegetarian fried bee hoon": "Char kway teow",
"Cili Kampung nasi ayam bakar": "Ikan bakar with rice and ulam",
"Cili Kampung asam pedas ikan with rice": "Steamed fish with rice and greens",
"Cili Kampung daging masak hitam with rice": "Rendang daging with rice",
"Cili Kampung sayur campur": "Stir-fried greens",
"Cong Caphe coconut coffee smoothie": "Chocolate milkshake",
"Cong Caphe Vietnamese iced coffee with condensed milk": "Kuih, 2 pieces",
"Cong Caphe egg coffee": "Latte, iced",
"Cong Caphe banh mi": "Bagel with cream cheese",
"Dave's ribeye steak": "Steak with vegetables",
"Dave's chicken chop": "Hainanese chicken chop",
"Dave's fish and chips": "Fish and chips",
"Dave's lamb shank": "Steak with vegetables",
"De Irrfan's nasi kerabu ayam": "Nasi kerabu with ayam percik",
"De Irrfan's laksa Johor": "Curry laksa",
"De Irrfan's roti john": "Roti canai kosong",
"Dragon-i Shanghai xiao long bao": "Xiao long bao",
"Dragon-i la mian in spicy soup": "Fish slice soup with tofu and tomato",
"Dragon-i dan dan noodles": "Fish slice soup with tofu and tomato",
"Dragon-i braised pork belly with rice": "Sweet and sour pork with rice",
"Dragon-i hot and sour soup": "Minced pork soup",
"Dragon-i deep-fried dumplings": "Pan-fried pork dumplings",
"Malatang, typical bowl": "Fish slice soup with tofu and tomato",
"Japanese curry rice (Dua)": "Chicken katsu curry rice",
"Empire Sushi salmon nigiri": "Sushi plates",
"Empire Sushi California roll": "Sushi plates",
"Empire Sushi salmon mentai roll": "Sushi plates",
"Empire Sushi chicken teriyaki don": "Sushi plates",
"Esquire Kitchen golden pumpkin noodles": "Wantan mee, dry",
"Esquire Kitchen salted fish fried rice": "Nasi goreng kampung",
"Esquire Kitchen sweet and sour fish with rice": "Sweet and sour pork with rice",
"Esquire Kitchen hokkien mee": "Hokkien mee (KL)",
"Fei Fan hot pot, per person": "Hot pot, typical meal",
"Fish & Co fish and chips": "Fish and chips",
"Fish & Co Fish in a Pan": "Fish and chips",
"Fish & Co seafood platter for one": "Fish and chips",
"Fish & Co New York fish and chips": "Fish and chips",
"Fish & Co clam chowder": "Minced pork soup",
"Fong Lye braised pork rice": "Sweet and sour pork with rice",
"Fong Lye Three-cup chicken set": "Sweet and sour pork with rice",
"Fong Lye beef noodles": "Boat noodles",
"Fong Lye pork chop rice": "Hainanese chicken chop",
"Food court chicken rice": "Chicken rice, steamed",
"Food court nasi campur, 2 dishes": "Economy rice: 1 meat, 2 veg",
"Food court char kuey teow": "Char kway teow",
"GIGI iced latte": "Latte, iced",
"GIGI butterscotch latte, iced": "Latte, iced",
"GIGI mocha, iced": "Mocha, iced",
"GO Unique nasi lemak ayam": "Nasi lemak biasa",
"GO Unique mee goreng": "Mee goreng mamak",
"Haidilao tomato broth hot pot, per person": "Hot pot, typical meal",
"Haidilao mala broth hot pot, per person": "Fish slice soup with tofu and tomato",
"Haidilao hand-pulled noodles": "Fish slice soup with tofu and tomato",
"Hansang bibimbap": "Bibimbap",
"Hansang kimchi jjigae": "Army stew (budae jjigae)",
"Hansang bulgogi set": "Bibimbap",
"Hansang japchae": "Bibimbap",
"Hokkaido Sushi salmon sashimi": "Salmon sashimi and rice",
"Hokkaido Sushi chirashi don": "Sushi plates",
"Hokkaido Sushi unagi don": "Sushi plates",
"Kim Gary baked cheese rice with chicken chop": "Hainanese chicken chop",
"Kim Gary pineapple bun with butter": "Pineapple bun with butter",
"Kim Gary French toast": "French toast",
"Kim Gary Hong Kong milk tea, iced": "Bubble milk tea",
"Kim Gary yin yong (coffee and tea)": "Bubble milk tea",
"Hotpot Kitchen, per person": "Hot pot, typical meal",
"House of Pok pork noodles, soup": "Wantan mee, dry",
"House of Pok char siew rice": "Char siu rice",
"HWC flat white": "Latte, iced",
"HWC iced mocha": "Mocha, iced",
"HWC chicken pie": "Apple pie",
"I Love Yoo! you tiao": "Pisang goreng",
"I Love Yoo! soy milk": "Mango lassi",
"I Love Yoo! you tiao with kaya": "Pisang goreng",
"India Gate chicken biryani": "Nasi briyani ayam",
"India Gate butter chicken with naan": "Butter chicken with naan",
"India Gate garlic naan": "Tandoori chicken with naan",
"India Gate tandoori chicken": "Tandoori chicken with naan",
"India Gate palak paneer": "Butter chicken with naan",
"India Gate mango lassi": "Mango lassi",
"Ippudo Shiromaru Classic": "Ramen, tonkotsu",
"Ippudo Akamaru Modern": "Ramen, tonkotsu",
"Ippudo Karaka Spicy ramen": "Ramen, tonkotsu",
"Ippudo chicken ramen": "Ramen, tonkotsu",
"Ippudo pork bun": "Char siu bao",
"Ippudo gyoza": "Pan-fried pork dumplings",
"Jardin earl grey milk tea": "Bubble milk tea",
"Jardin iced latte": "Latte, iced",
"Jatujak boat noodles": "Boat noodles",
"Jatujak pad kra pao with egg": "Pad kra pao with rice and egg",
"Jatujak moo ping skewers": "Satay ayam",
"Jatujak mango sticky rice": "Mango sticky rice",
"Jipinhe scone, plain": "Scone with jam and cream",
"Jipinhe chocolate scone": "Scone with jam and cream",
"Johnny's tom yum steamboat, per person": "Hot pot, typical meal",
"Johnny's Thai fried rice": "Nasi goreng kampung",
"Johnny's chicken chop": "Hainanese chicken chop",
"K Fry Yangnyeom chicken": "Korean fried chicken, soy garlic",
"K Fry soy garlic chicken": "Korean fried chicken, soy garlic",
"K Fry Spicy Black Garlic Chicken": "Korean fried chicken, soy garlic",
"K Fry kimchi fried rice": "Nasi goreng kampung",
"K Fry rabokki": "Tteokbokki",
"K Fry Berry-Matcha Mochi Bingsoo": "Injeolmi bingsu",
"KAKI 5 pan mee": "Pan mee, soup",
"KAKI 5 curry mee": "Curry laksa",
"Kampung Dining nasi kerabu": "Nasi kerabu with ayam percik",
"Kampung Dining nasi lemak ayam rendang": "Nasi lemak biasa",
"Kampung Dining asam laksa": "Asam laksa",
"Kenangan Signature coffee latte, iced": "Latte, iced",
"Kenangan Avocado coffee": "Latte, iced",
"Kenangan Black Aren latte": "Latte, iced",
"Ko Hyang dolsot bibimbap": "Bibimbap",
"Ko Hyang sundubu jjigae": "Army stew (budae jjigae)",
"Ko Hyang dakgalbi": "Korean BBQ, typical meal",
"Kyochon Soy Garlic wings": "Korean fried chicken, soy garlic",
"Kyochon Red Original wings": "Korean fried chicken, soy garlic",
"Kyochon Honey Original drumsticks": "Korean fried chicken, soy garlic",
"Kyochon Salsal chicken strips": "Chicken nuggets",
"Laem Charoen deep-fried seabass with fish sauce": "Steamed fish with rice and greens",
"Laem Charoen steamed seabass with lime and chili": "Steamed fish with rice and greens",
"Laem Charoen crab fried rice": "Nasi goreng kampung",
"Laem Charoen tom yum seafood": "Tom yum soup with rice",
"Little Cravings nasi lemak with fried chicken": "Nasi lemak biasa",
"Little Cravings char kuey teow": "Char kway teow",
"Little Cravings Penang assam laksa": "Asam laksa",
"Little Nyonya nasi lemak Nyonya": "Nasi lemak biasa",
"Little Nyonya Nyonya laksa": "Curry laksa",
"Little Nyonya ayam buah keluak with rice": "Green curry chicken with rice",
"Long Jing Dongpo pork with rice": "Sweet and sour pork with rice",
"Long Jing green tea chicken": "Bubble milk tea",
"Long Jing tofu with minced pork": "Stir-fried greens",
"LOVNS banh mi": "Bagel with cream cheese",
"LOVNS iced latte": "Latte, iced",
"Luretta margherita pizza": "Pizza",
"Luretta truffle mushroom pasta": "Spaghetti carbonara",
"Luretta tiramisu": "Tiramisu",
"Madam Kwan's nasi lemak": "Nasi lemak biasa",
"Madam Kwan's nasi bojari": "Nasi briyani ayam",
"Madam Kwan's char kuey teow": "Char kway teow",
"Madam Kwan's Nasi Ayam Gepuk": "Nasi ayam penyet",
"Madam Kwan's assam laksa": "Asam laksa",
"Madam Kwan's cendol": "Cendol",
"Makii Makii onigiri, tuna mayo": "Sushi plates",
"Makii Makii salmon hand roll": "Sushi plates",
"Marco margherita pizza": "Pizza",
"Marco carbonara": "Spaghetti carbonara",
"Marco tiramisu": "Tiramisu",
"Mee tarik beef soup": "Minced pork soup",
"Mee tarik fried with chicken": "Mee goreng mamak",
"Mee tarik dry with minced beef": "Wantan mee, dry",
"Melaka chicken rice balls": "Chicken rice, steamed",
"Melaka Nyonya asam pedas fish with rice": "Steamed fish with rice and greens",
"Melaka cendol with gula Melaka": "Cendol",
"Menya Musashi tsukemen": "Ramen, tonkotsu",
"Menya Musashi black ramen": "Ramen, tonkotsu",
"Menya Musashi red spicy ramen": "Ramen, tonkotsu",
"MODU Korean corn dog": "Pisang goreng",
"MODU tteokbokki": "Tteokbokki",
"Monster Curry pork katsu curry": "Chicken katsu curry rice",
"Monster Curry chicken katsu curry": "Green curry chicken with rice",
"Monster Curry omu curry rice": "Chicken katsu curry rice",
"Mr Fish fish head noodle soup": "Fish slice soup with tofu and tomato",
"Mr Fish sliced fish noodle soup": "Fish slice soup with tofu and tomato",
"Mr Fish seafood noodles, milky broth": "Fish slice soup with tofu and tomato",
"Mr Tuk Tuk pad thai": "Pad thai",
"Mr Tuk Tuk tom yum fried rice": "Nasi goreng kampung",
"Mr Tuk Tuk Thai iced tea": "Bubble milk tea",
"NakNak bulgogi kimbap": "Sushi plates",
"NakNak ramyeon with egg": "Ramen, shoyu",
"Nam Heong white coffee, iced": "Latte, iced",
"Nam Heong egg tart": "Egg tart",
"Nam Heong Ipoh chicken hor fun": "Ipoh hor fun",
"Nam Heong curry mee": "Curry laksa",
"Nippori matcha latte": "Matcha latte, iced",
"Nippori matcha soft serve": "Ice cream",
"Noodle Shack chili pan mee": "Pan mee, soup",
"Noodle Shack dry wantan noodles": "Wantan mee, dry",
"Nyonya Colors kuih lapis": "Kuih, 2 pieces",
"Nyonya Colors onde-onde": "Kuih, 2 pieces",
"Nyonya Colors nasi lemak": "Nasi lemak biasa",
"Nyonya Colors curry puff": "Curry puff",
"Onigirazu, chicken katsu": "Sushi plates",
"Onigirazu, salmon and egg": "Sushi plates",
"Oversea char siew": "Char siu rice",
"Oversea yang chow fried rice": "Nasi goreng kampung",
"Oversea dim sum, 3 baskets": "Dim sum breakfast",
"Paris Baguette croissant": "Butter croissant",
"Paris Baguette strawberry cream cake": "Chocolate cake",
"Paris Baguette ham and cheese sandwich": "Cheeseburger",
"Paris Baguette soboro bread": "Char siu bao",
"Park's Bagels plain bagel with cream cheese": "Bagel with cream cheese",
"Park's Bagels smoked salmon bagel": "Bagel with cream cheese",
"Park's Bagels egg and cheese bagel": "Bagel with cream cheese",
"Pasar Eatai char kuey teow": "Char kway teow",
"Pasar Eatai Hokkien mee": "Hokkien mee (KL)",
"Pasar Eatai satay, 6 sticks": "Satay ayam",
"Penang chendul": "Cendol",
"Penang chendul with red beans and pulut": "Cendol",
"Penang ais kacang": "Ais kacang (ABC)",
"Petite Maison pain au chocolat": "Butter croissant",
"Petite Maison sourdough sandwich": "Cheeseburger",
"Pho Vietz beef pho": "Pho",
"Pho Vietz chicken pho": "Pho",
"Pho Vietz fresh spring rolls": "Pisang goreng",
"Pho Vietz fried spring rolls": "Pisang goreng",
"Pho Vietz bun bo hue": "Pho",
"Pho Vietz Vietnamese drip coffee": "Pho",
"Porridge Time chicken porridge": "Chicken porridge",
"Porridge Time fish porridge": "Chicken porridge",
"Porridge Time pork porridge with century egg": "Chicken porridge",
"Pu Tien Heng Hwa fried bee hoon": "Char kway teow",
"Pu Tien braised pork belly with buns": "Char siu bao",
"Pu Tien lor mee": "Lor mee",
"Pu Tien stir-fried yam": "Stir-fried greens",
"Pao Xiang bak kut teh set": "Bak kut teh with rice",
"Pao Xiang dry bak kut teh": "Bak kut teh with rice",
"Peranakan Place ayam pongteh with rice": "Green curry chicken with rice",
"Peranakan Place babi pongteh with rice": "Green curry chicken with rice",
"Rich Kopitiam kaya toast set": "Kaya toast set",
"Rich Kopitiam curry mee": "Curry laksa",
"Rich Kopitiam nasi lemak": "Nasi lemak biasa",
"Robataya robatayaki set": "Korean BBQ, typical meal",
"Robataya salmon teriyaki set": "Chicken teriyaki don",
"Rocku yakiniku beef set, per person": "Korean BBQ, typical meal",
"Samtai nasi lemak": "Nasi lemak biasa",
"Samtai kaya butter toast": "Kaya toast set",
"Samtai Hainan chicken chop rice": "Hainanese chicken chop",
"Se'Indonesia nasi ayam penyet": "Nasi ayam penyet",
"Se'Indonesia bakso": "Minced pork soup",
"Se'Indonesia gado-gado": "Som tam",
"Sekinchan fish and chips": "Fish and chips",
"Sekinchan grilled fish rice": "Ikan bakar with rice and ulam",
"Shabu-Yo shabu shabu, per person": "Hot pot, typical meal",
"Shake Shack ShackBurger single": "Cheeseburger",
"Shake Shack Double ShackBurger": "Cheeseburger",
"Shake Shack Chick'n Shack": "Cheeseburger",
"Shake Shack crinkle cut fries": "French fries",
"Shake Shack cheese fries": "French fries",
"Shake Shack vanilla shake": "Chocolate milkshake",
"Jjajangmyeon": "Jajangmyeon",
"Budae jjigae, per person": "Army stew (budae jjigae)",
"Jjamppong": "Jajangmyeon",
"Tangsuyuk (sweet and sour pork)": "Sweet and sour pork with rice",
"Since 1939 claypot chicken rice": "Claypot chicken rice",
"Since 1939 Hokkien mee": "Hokkien mee (KL)",
"Soup Restaurant Samsui ginger chicken": "Minced pork soup",
"Soup Restaurant double-boiled soup of the day": "Minced pork soup",
"Starbucks caffe latte, grande": "Latte, iced",
"Starbucks caramel macchiato, grande": "Latte, iced",
"Starbucks java chip Frappuccino, grande": "Chocolate milkshake",
"Starbucks matcha latte, grande": "Matcha latte, iced",
"Starbucks butter croissant": "Butter croissant",
"Strangers iced latte": "Latte, iced",
"Strangers brunch plate": "Big breakfast",
"Stuff'd kebab wrap, chicken": "Bagel with cream cheese",
"Stuff'd burrito": "Bagel with cream cheese",
"Sushi Zanmai salmon sashimi": "Salmon sashimi and rice",
"Sushi Zanmai salmon nigiri": "Sushi plates",
"Sushi Zanmai chawanmushi": "Sushi plates",
"Sushi Zanmai unagi don": "Sushi plates",
"Sushi Zanmai salmon don": "Sushi plates",
"Sushiro salmon nigiri": "Sushi plates",
"Sushiro thick cut salmon belly": "Salmon sashimi and rice",
"Sushiro grilled eel nigiri": "Sushi plates",
"Sushiro tuna nigiri": "Sushi plates",
"Sushiro typical meal": "Sushi plates",
"Sushiya salmon don": "Sushi plates",
"Sushiya chicken katsu don": "Sushi plates",
"Tai Er sauerkraut fish, per person": "Fish slice soup with tofu and tomato",
"Tai Ho Jiak lu rou fan": "Sweet and sour pork with rice",
"Tai Ho Jiak Taiwanese fried chicken": "Korean fried chicken, soy garlic",
"Tanyu grilled fish, per person": "Steamed fish with rice and greens",
"Teh Tarik Club teh tarik": "Teh tarik",
"Teh Tarik Club roti canai with curry": "Roti canai kosong",
"Teh Tarik Nation teh tarik": "Teh tarik",
"Teh Tarik Nation nasi lemak": "Nasi lemak biasa",
"Teochew lor mee": "Lor mee",
"Teochew braised duck rice": "Roast duck rice",
"Teppanyaki chicken set": "Chicken teriyaki don",
"Teppanyaki salmon set": "Chicken teriyaki don",
"TGI Fridays Jack Daniel's glazed chicken": "Doughnut",
"TGI Fridays loaded potato skins": "French fries",
"TGI Fridays Fridays cheeseburger": "Cheeseburger",
"TGI Fridays sesame jack chicken strips": "Chicken nuggets",
"Thai chicken rice (khao man gai)": "Chicken rice, steamed",
"Thai Hou Sek tom yum noodles": "Pho",
"Thai Hou Sek basil pork rice": "Pad kra pao with rice and egg",
"Coffee Bean Ice Blended, mocha, regular": "Chocolate milkshake",
"Coffee Bean vanilla latte, iced, regular": "Latte, iced",
"Coffee Bean chicken pie": "Apple pie",
"Daily Greens grain bowl with chicken": "Chicken caesar salad",
"Daily Greens salmon salad": "Salmon sashimi and rice",
"Fish Bowl poke bowl, salmon": "Salmon sashimi and rice",
"Fish Bowl poke bowl, tuna": "Salmon sashimi and rice",
"Tim Hortons Iced Capp": "Chocolate milkshake",
"Tim Hortons Timbits, chocolate glazed": "Doughnut",
"Tim Hortons double double coffee": "Latte, iced",
"Tim Hortons Farmer's breakfast wrap": "Bagel with cream cheese",
"Ma Maison rosu katsu set": "Tonkatsu set",
"Ma Maison hire katsu set": "Tonkatsu set",
"Ma Maison katsu curry": "Chicken katsu curry rice",
"Tous les Jours walnut pound cake": "Chocolate cake",
"Tous les Jours sausage bread": "Char siu bao",
"Tous les Jours fresh cream cake": "Chocolate cake",
"Wang Donkas pork cutlet": "Tonkatsu set",
"Wang Donkas cheese donkas": "Tonkatsu set",
"Wanhoi wantan noodles": "Wantan mee, dry",
"Wanhoi prawn wantan soup": "Fried wantan",
"Whisk flat white": "Latte, iced",
"Whisk big breakfast": "Big breakfast",
"Whisk eggs Benedict": "Big breakfast",
"Wok Hey egg fried rice with chicken": "Nasi goreng kampung",
"Wok Hey black pepper beef fried rice": "Nasi goreng kampung",
"Wok Hey fried udon": "Char kway teow",
"Wong Kok baked pork chop rice": "Baked pork chop rice",
"Wong Kok satay beef noodles": "Boat noodles",
"Wong Kok pineapple bun with butter": "Pineapple bun with butter",
"Wong Kok milk tea, iced": "Bubble milk tea",
"Wonjo Buanjib pork belly BBQ set, per person": "Korean BBQ, typical meal",
"WPS Cafe spaghetti aglio olio": "Spaghetti carbonara",
"WPS Cafe iced latte": "Latte, iced",
"Pork and chive dumplings, boiled": "Pan-fried pork dumplings",
"Pan-fried pork dumplings": "Pan-fried pork dumplings",
"Hot and sour soup": "Minced pork soup",
"Yakiniku Kuro wagyu set, per person": "Steak with vegetables",
"Yat Yat Seng wantan mee": "Wantan mee, dry",
"Yat Yat Seng roast pork rice": "Char siu rice",
"Yuan Room dim sum, 3 baskets": "Dim sum breakfast",
"Yuan Room roast duck": "Roast duck rice",
"Yunnan crossing-the-bridge noodles": "Pho",
"Yunnan spicy rice noodles": "Pho",
"Yun Nans steam pot chicken soup": "Hot pot, typical meal",
"Yun Nans rice noodles with minced pork": "Pho",
"Zok bamboo noodles with char siu": "Wantan mee, dry",
"ZUS Spanish latte, iced": "Latte, iced",
"ZUS butter croissant": "Butter croissant",
"Come True cafe brunch pasta": "Spaghetti carbonara",
"Asador grilled steak, per person": "Korean BBQ, typical meal",
"Kim Lian Kee Hokkien mee": "Hokkien mee (KL)",
"Soong Kee beef ball noodles, dry": "Wantan mee, dry",
"Duck egg char kuey teow": "Char kway teow",
"Klang bak kut teh": "Bak kut teh with rice",
"Hon Kee fish porridge": "Chicken porridge",
"Imbi pork noodles, soup": "Wantan mee, dry",
"The Bean nasi lemak": "Nasi lemak biasa",
"Masala King mutton biryani": "Nasi briyani ayam",
"Tutti Frutti frozen yogurt": "Mango lassi",
"Grand Harbour dim sum, per person": "Dim sum breakfast",
"Al-Amar Express chicken shawarma wrap": "Bagel with cream cheese",
"FWF hot pot, per person": "Hot pot, typical meal",
"Mengs Kee claypot chicken rice": "Claypot chicken rice",
"Corks Out steak with fries": "Steak with vegetables",
"Luk Yu dim sum, per person": "Dim sum breakfast",
"Shook! grilled wagyu": "Steak with vegetables",
"Enak KL nasi lemak with rendang": "Nasi lemak biasa",
"Baked pork chop rice": "Baked pork chop rice",
"Pineapple bun with butter": "Pineapple bun with butter",
"Ipoh hor fun": "Ipoh hor fun",
"Lor mee": "Lor mee",
"Pad kra pao with rice and egg": "Pad kra pao with rice and egg",
"Waffle with ice cream": "Waffle with ice cream",
"DIN xiao long bao": "Xiao long bao",
"DIN pork chop fried rice": "Nasi goreng kampung",
"Little Penang curry mee": "Curry laksa",
"Little Penang char kway teow": "Char kway teow",
"Little Penang nasi lemak": "Nasi lemak biasa",
"Little Penang asam laksa": "Asam laksa",
"Bumbu Desa ayam penyet with rice": "Nasi ayam penyet",
"Bumbu Desa nasi campur": "Economy rice: 1 meat, 2 veg",
"Aseana nasi lemak": "Nasi lemak biasa",
"Chakri Palace pineapple fried rice": "Nasi goreng kampung",
"Chakri Palace green curry": "Green curry chicken with rice",
"Para Thai tom yum noodles": "Pho",
"Rén Peking duck, per person": "Roast duck rice",
"Rén la mian with minced pork": "Fish slice soup with tofu and tomato",
"Ki Ki dry noodles with scallion oil": "Wantan mee, dry",
"Koryo-Won bulgogi set": "Bibimbap",
"Koryo-Won Korean BBQ, per person": "Korean BBQ, typical meal",
"Sakana sushi set": "Sushi plates",
"Chinoz pasta carbonara": "Spaghetti carbonara",
"Delirium Belgian mussels with fries": "Butter prawns",
"Maria's Signature Spanish paella": "Nasi goreng kampung",
"Melur & Thyme nasi lemak": "Nasi lemak biasa",
"Open House nasi lemak with ayam goreng": "Nasi lemak biasa",
"O'Briens chicken sandwich": "Cheeseburger",
"Acme burger": "Cheeseburger",
"Acme iced latte": "Latte, iced",
"Bape Cafe latte": "Latte, iced",
"Eight Ounce iced latte": "Latte, iced",
"Bisou cupcake": "Chocolate cake",
"Hazukido croissant": "Butter croissant",
"Hui Lau Shan mango pomelo sago": "Tau fu fah, less syrup",
"Salon Du Chocolat hot chocolate": "Chocolate bar",
"Yomie's purple rice yogurt": "Mango lassi",
"B& iced latte": "Latte, iced",
"Ahh Yum nasi bunga telang": "Nasi lemak biasa",
"Food court wantan mee": "Wantan mee, dry",
"Eatomo chicken teriyaki bento": "Chicken teriyaki don",
"Fuel Shack Jumbo Smoothie Bowl": "Chocolate milkshake",
"Fuel Shack Big Shack burger": "Cheeseburger",
"JJ chili pan mee": "Pan mee, soup",
"Kagura chicken paitan ramen": "Ramen, tonkotsu",
"Ken-Chan katsu curry rice": "Chicken katsu curry rice",
"Nathan's Famous hot dog": "Pretzel, original",
"Nathan's Famous cheese fries": "French fries",
"Milano pizza, 2 slices": "Pizza",
"Milano spaghetti bolognese": "Spaghetti carbonara",
"Saravanaa Bhavan masala dosa": "Thosai with dhal and chutney",
"Saravanaa Bhavan South Indian meals (banana leaf)": "Nasi briyani ayam",
"Saravanaa Bhavan idli": "Thosai with dhal and chutney",
"Shihlin XXL crispy chicken": "Korean fried chicken, soy garlic",
"Shihlin oyster mee sua": "Mee hoon soup with fish balls",
"Shihlin popcorn chicken": "Chicken nuggets",
"Sizzles sizzling chicken chop": "Hainanese chicken chop",
"Warung Prima nasi kunyit with chicken": "Nasi lemak biasa",
"Hameed nasi kandar": "Nasi briyani ayam",
"Big Pan pan mee": "Pan mee, soup",
"Pizza Hut Super Supreme pan pizza": "Pizza",
"Pizza Hut Hawaiian Chicken pan pizza": "Pizza",
"Pizza Hut pepperoni pan pizza": "Pizza",
"Pizza Hut Chicken Supreme thin crust": "Pizza",
"Pizza Hut cheesy stuffed crust pizza": "Pizza",
"Pizza Hut chicken wings": "Korean fried chicken, soy garlic",
"Pizza Hut garlic bread": "Pizza",
"Pizza Hut cheesy baked spaghetti, bolognese": "Spaghetti carbonara",
"Pizza Hut mushroom soup": "Pizza",
"Domino's Chicken Pepperoni pizza, hand tossed": "Pizza",
"Domino's Domino's Deluxe pizza, hand tossed": "Pizza",
"Domino's Hawaiian Paradise pizza": "Pizza",
"Carl's Jr Famous Star with cheese": "Cheeseburger",
"Carl's Jr Western Bacon Cheeseburger": "Cheeseburger",
"Carl's Jr Guacamole Bacon Burger": "Cheeseburger",
"Carl's Jr crispy chicken sandwich": "Cheeseburger",
"Carl's Jr onion rings": "French fries",
"Marrybrown fried chicken, 2 pieces": "KFC Original Recipe, thigh",
"Marrybrown MB burger": "Cheeseburger",
"Marrybrown nasi lemak MB": "Nasi lemak biasa",
"Marrybrown chicken porridge": "Chicken porridge",
"Jollibee Chickenjoy, 1 piece with rice": "KFC Original Recipe, thigh",
"Jollibee Jolly Spaghetti": "Spaghetti bolognese",
"Jollibee Yumburger": "Cheeseburger",
"Jollibee peach mango pie": "Apple pie",
"Dunkin' glazed donut": "Doughnut",
"Dunkin' Boston Kreme donut": "Doughnut",
"Dunkin' chocolate frosted donut": "Doughnut",
"Dunkin' Munchkins": "Doughnut",
"Krispy Kreme Original Glazed": "Doughnut",
"Krispy Kreme chocolate iced glazed": "Doughnut",
"Krispy Kreme filled doughnut": "Doughnut",
"Cinnabon Classic Roll": "Cinnabon Classic Roll",
"Cinnabon MiniBon": "Cinnabon Classic Roll",
"Cinnabon Chocobon": "Cinnabon Classic Roll",
"Beard Papa original cream puff": "Beard Papa original cream puff",
"Beard Papa chocolate cream puff": "Beard Papa original cream puff",
"Beard Papa eclair": "Beard Papa original cream puff",
"Famous Amos chocolate chip cookies": "Chocolate chip cookie",
"Ben's Cookies milk chocolate cookie": "Chocolate chip cookie",
"Haagen-Dazs single scoop": "Ice cream",
"Haagen-Dazs fondue for one": "Ice cream",
"Lindt Lindor truffles": "Chocolate bar",
"Lindt hot chocolate": "Chocolate bar",
"Chateraise strawberry shortcake": "Chocolate cake",
"Chateraise cream puff": "Beard Papa original cream puff",
"Komugi cheese tart": "Egg tart",
"Komugi curry bread": "Char siu bao",
"Mille crepe, original": "Chocolate cake",
"Mille crepe, matcha": "Chocolate cake",
"Garrett CaramelCrisp popcorn": "Popcorn, caramel",
"Garrett Chicago Mix popcorn": "Popcorn, caramel",
"Happy Potato tornado potato": "French fries",
"Happy Potato cheese potato balls": "French fries",
"Hot & Roll chicken roll": "Bagel with cream cheese",
"Crispy popiah": "Pisang goreng",
"Bibi's popiah, fresh": "Popiah, fresh",
"LFC fried chicken, 2 pieces": "KFC Original Recipe, thigh",
"Boost Juice Mango Magic": "Soft drink",
"Boost Juice Berry Crush": "Soft drink",
"Juice Works Mango Tango": "Soft drink",
"Juice Works watermelon juice": "Soft drink",
"HEYTEA cheese grape": "Bubble milk tea",
"HEYTEA jasmine milk tea": "Bubble milk tea",
"ChaTraMue Thai tea": "Bubble milk tea",
"ChaTraMue Thai green tea": "Bubble milk tea",
"ShuYi grass jelly milk tea": "Bubble milk tea",
"Oat Story oat latte": "Latte, iced",
"Claypot white coffee": "Latte, iced",
"Frozen yogurt with toppings": "Mango lassi",
"Yolé frozen yogurt": "Mango lassi",
"Snowflake taro balls dessert": "Cendol",
"Sugirl durian mango dessert": "Cendol",
"Shantea grass jelly dessert": "Cendol",
"Pobbyice bingsu": "Injeolmi bingsu",
"Herbal Farmer herbal jelly": "Big breakfast",
"Roasted Gong pear drink": "Soft drink",
"Paradise Dynasty 8-flavour xiao long bao": "Xiao long bao",
"Paradise Dynasty la mian with braised pork": "Fish slice soup with tofu and tomato",
"Paradise Dynasty dan dan noodles": "Fish slice soup with tofu and tomato",
"Paradise Dynasty fried rice with shrimp": "Nasi goreng kampung",
"Grand Taipei braised pork rice": "Sweet and sour pork with rice",
"Grand Taipei beef noodle soup": "Minced pork soup",
"Grand Taipei popcorn chicken": "Chicken nuggets",
"Amarin green curry chicken": "Green curry chicken with rice",
"Amarin tom yum goong": "Tom yum soup with rice",
"Amarin pad thai": "Pad thai",
"Tanba yakiniku set, per person": "Korean BBQ, typical meal",
"Palsaik 8-flavour pork belly set, per person": "Korean BBQ, typical meal",
"Palsaik kimchi stew": "Army stew (budae jjigae)",
"BBQ Town grill set, per person": "Korean BBQ, typical meal",
"Xiao Long Kan mala hot pot, per person": "Fish slice soup with tofu and tomato",
"Mala mini hotpot": "Fish slice soup with tofu and tomato",
"Fei Fei Crab spicy crab pot, per person": "Butter prawns",
"Peng Chu Hunan chili pork": "Steamed fish with rice and greens",
"Peng Chu steamed fish with chili": "Steamed fish with rice and greens",
"Souper Tang double-boiled chicken soup": "Minced pork soup",
"MyIPOH Ipoh chicken hor fun": "Ipoh hor fun",
"MyIPOH bean sprout chicken": "Chicken rice, steamed",
"MyIPOH white coffee": "Latte, iced",
"NALE nasi lemak ayam goreng": "Nasi lemak biasa",
"NALE nasi lemak rendang daging": "Nasi lemak biasa",
"Nasi Lemak Shop nasi lemak ayam berempah": "Nasi lemak biasa",
"Nasi kandar with fried chicken and curry": "Nasi briyani ayam",
"Kayu roti canai with dhal": "Roti canai kosong",
"Kayu mee goreng mamak": "Mee goreng mamak",
"Ali Muthu nasi lemak": "Nasi lemak biasa",
"Ali Muthu roti bakar set": "Kaya toast set",
"Ayam gepuk with rice": "Nasi ayam penyet",
"Nasi Arab mandi with chicken": "Nasi briyani ayam",
"Lalezar mixed grill kebab": "Satay ayam",
"Lalezar chicken shawarma wrap": "Bagel with cream cheese",
"Kiku-Zakura salmon teriyaki set": "Chicken teriyaki don",
"Kiku-Zakura tempura udon": "Udon in soup",
"Kamakura chicken katsu don": "Tonkatsu set",
"Bankara ramen": "Ramen, tonkotsu",
"Bankara tsukemen": "Ramen, tonkotsu",
"MyeongDong topokki": "Tteokbokki",
"MyeongDong Korean fried chicken": "Korean fried chicken, soy garlic",
"MyeongDong ramyeon": "Ramen, shoyu",
"Halla Holla Korean corn dog": "Pisang goreng",
"Sin Man Kee baked pork chop rice": "Baked pork chop rice",
"Sin Man Kee pineapple bun with butter": "Pineapple bun with butter",
"Sin Man Kee Hong Kong milk tea": "Bubble milk tea",
"Mak's Chee wonton noodle soup": "Fried wantan",
"Mak's Chee beef brisket noodles": "Pho",
"Omega pork noodles, soup": "Wantan mee, dry",
"Go Noodle signature mixed pork noodle soup": "Minced pork soup",
"Good Chen chicken rice": "Chicken rice, steamed",
"Lo Hong Ka chicken rice": "Chicken rice, steamed",
"Kwan Kee claypot rice": "Claypot chicken rice",
"Oriental Kopi kaya butter toast": "Kaya toast set",
"Oriental Kopi white coffee": "Latte, iced",
"Oriental Kopi egg tart": "Egg tart",
"Travelling Duck roast duck rice": "Roast duck rice",
"Dao kampung chicken rice": "Chicken rice, steamed",
"Antipodean big breakfast": "Big breakfast",
"Antipodean flat white": "Latte, iced",
"Ben's fish and chips": "Fish and chips",
"Ben's chicken pie": "Apple pie",
"The Barn fried chicken": "KFC Original Recipe, thigh",
"LingLing LongLong fusion rice bowl": "Economy rice: 1 meat, 2 veg",
"Christine's bun": "Char siu bao",
"Yin's sourdough sandwich": "Cheeseburger",
"Founders Bakery bun": "Char siu bao",
"Lapisan kek lapis": "Chocolate cake",
"Just Good Coffee iced latte": "Latte, iced",
"Nononsense iced latte": "Latte, iced",
"Kyo ha matcha latte": "Matcha latte, iced",
"Kyo ha matcha parfait": "Matcha latte, iced",
"dipndip chocolate crepe": "Waffle with ice cream",
"dipndip waffle with chocolate": "Waffle with ice cream",
"Creme de la creme fruit tart": "Fruit tart",
"Sukiya gyudon, regular": "Udon in soup",
"Sukiya cheese gyudon": "Udon in soup",
"Sukiya curry rice": "Chicken katsu curry rice",
"Jungle House chicken rice": "Chicken rice, steamed",
"B.O.D steamed bun with meat": "Char siu bao",
"Omiyage Japanese cheese tart": "Egg tart",
"Shae.co bagel sandwich": "Cheeseburger",
"Rightside burger": "Cheeseburger",
"Apollo rice set with 2 dishes": "Economy rice: 1 meat, 2 veg",
"Food hall mixed rice": "Economy rice: 1 meat, 2 veg",
"Squeezby fresh juice": "Soft drink",
"Red Box set lunch, chicken chop rice": "Hainanese chicken chop",
"Ke Ren Lai Hakka lei cha rice": "Economy rice: 1 meat, 2 veg",
"Ichiban tonkotsu ramen": "Ramen, tonkotsu",
"Mr Dakgalbi cheese dakgalbi, per person": "Korean BBQ, typical meal",
"Nong Geng Ji pepper fried pork": "Economy rice: 1 meat, 2 veg",
"Five Guys cheeseburger": "Cheeseburger",
"Five Guys little cheeseburger": "Cheeseburger",
"Five Guys bacon cheeseburger": "Cheeseburger",
"Five Guys Cajun fries, little": "French fries",
"Five Guys milkshake": "Chocolate milkshake",
"Gong Cha milk tea with pearls": "Bubble milk tea",
"Gong Cha milk foam green tea": "Bubble milk tea",
"Gong Cha brown sugar fresh milk with pearls": "Bubble milk tea",
"Gong Cha mango yakult": "Bubble milk tea",
"Luckin coconut latte, iced": "Latte, iced",
"Luckin Americano, iced": "Kopi C",
"Luckin velvet latte": "Latte, iced",
"Coolblog original milk tea with pearls": "Bubble milk tea",
"Coolblog ice blended chocolate": "Chocolate bar",
"Chicha San Chen Dong Ding oolong milk tea": "Bubble milk tea",
"Arabica Spanish latte, iced": "Latte, iced",
"Arabica cafe latte": "Latte, iced",
"SF Coffee cafe mocha": "Mocha, iced",
"SF Coffee chicken pie": "Apple pie",
"illy cappuccino": "Latte, iced",
"TWG tea, plain": "Kopi C",
"TWG macarons": "Chocolate chip cookie",
"TWG scone with jam and tea jelly": "Scone with jam and cream",
"Dome chicken pie": "Apple pie",
"Dome Dome burger": "Cheeseburger",
"Dome iced mocha": "Mocha, iced",
"PAUL croissant": "Butter croissant",
"PAUL croque monsieur": "Bagel with cream cheese",
"PAUL eclair": "Beard Papa original cream puff",
"Rotiboy coffee bun": "Char siu bao",
"Cold Stone Like It ice cream creation": "Ice cream",
"Gindaco takoyaki, original": "Takoyaki",
"Gindaco takoyaki, cheese mentai": "Takoyaki",
"Potato Corner flavoured fries, regular": "French fries",
"Tokyo Milk Cheese Factory cheese cookies": "Chocolate chip cookie",
"Royce Nama chocolate": "Chocolate bar",
"Godiva chocolixir": "Chocolate milkshake",
"Godiva soft serve cone": "Ice cream",
"Venchi gelato, 2 flavours in a cup": "Ice cream",
"Cioccolati Italiani gelato cone": "Ice cream",
"Harriston hot chocolate": "Chocolate bar",
"Melvados creme brulee cheesecake": "Cheesecake",
"Winky Walnut walnut cakes": "Chocolate chip cookie",
"Croissant taiyaki, custard": "Butter croissant",
"Honey Ring donut": "Doughnut",
"adough donut": "Doughnut",
"Cookiecrumbs cookie": "Chocolate chip cookie",
"Ning Ji hand-pounded lemon tea": "Bubble milk tea",
"Lam Heung Ling fragrant lemon tea": "Bubble milk tea",
"O'Cha matcha latte": "Matcha latte, iced",
"108 Matcha Saro matcha soft serve": "Ice cream",
"Matcha Hero matcha latte": "Matcha latte, iced",
"Loka iced latte": "Latte, iced",
"Hub Coffee iced latte": "Latte, iced",
"Ralph's iced latte": "Latte, iced",
"Cafe Kitsune iced latte": "Latte, iced",
"Cafe Kitsune fox shortbread": "Chocolate chip cookie",
"fRost n Bites frozen yogurt": "Mango lassi",
"Chinese egg roll biscuits": "Chocolate chip cookie",
"Eureka popcorn": "Popcorn, caramel",
"Leekor Leekor drink": "Bubble milk tea",
"Mama Kimbap tuna mayo kimbap": "Sushi plates",
"Mama Kimbap bulgogi kimbap": "Sushi plates",
"Oloiya Korean snack": "Tteokbokki",
"Tan Ngan Lo herbal tea": "Soft drink",
"Taimal herbal tea": "Soft drink",
"Nespresso espresso": "Kopi C",
"Putien Lor mee": "Lor mee",
"Putien Heng Hwa fried bee hoon": "Char kway teow",
"Putien sweet and sour pork with lychee": "Sweet and sour pork with rice",
"Putien stir-fried yam": "Stir-fried greens",
"Putien Bian Rou soup (meat dumplings)": "Pan-fried pork dumplings",
"Kampachi bento set": "Chicken teriyaki don",
"Kampachi chirashi don": "Salmon sashimi and rice",
"Kampachi gyu teriyaki set": "Chicken teriyaki don",
"Uncle Roger egg fried rice": "Nasi goreng kampung",
"Uncle Roger fried chicken egg fried rice": "Nasi goreng kampung",
"Santouka shio ramen": "Ramen, tonkotsu",
"Santouka toroniku ramen": "Ramen, tonkotsu",
"Aori tonkotsu ramen": "Ramen, tonkotsu",
"Ichi Zen sashimi set": "Salmon sashimi and rice",
"Maisen kurobuta tonkatsu set": "Tonkatsu set",
"Maisen katsu sando": "Tonkatsu set",
"Donkai katsu don": "Tonkatsu set",
"Takeno udon set": "Udon in soup",
"Shojikiya beef sukiyaki set": "Hot pot, typical meal",
"Yamazo chicken teriyaki don": "Chicken teriyaki don",
"Yu Yan dim sum lunch, per person": "Dim sum breakfast",
"Dolly Dim Sum har gow": "Dim sum breakfast",
"Dolly Dim Sum siu mai": "Dim sum breakfast",
"Dolly Dim Sum char siu bun": "Char siu bao",
"Dolly Dim Sum chee cheong fun with prawns": "Dim sum breakfast",
"Dai Cha Dim dim sum, per person": "Dim sum breakfast",
"Canton Court roast duck": "Roast duck rice",
"Grand Imperial roast pork and duck rice": "Roast duck rice",
"Xu's Cuisine Hunan chili fish": "Steamed fish with rice and greens",
"Siwawa Guizhou wraps": "Bagel with cream cheese",
"Wuliguan hot pot, per person": "Hot pot, typical meal",
"Yang Guo Fu mala tang": "Hot pot, typical meal",
"Siong Tong Gai steamed chicken rice": "Chicken rice, steamed",
"JYNNS steamed chicken rice": "Chicken rice, steamed",
"On Lee wonton noodles": "Fried wantan",
"Hot Crush pork noodles": "Wantan mee, dry",
"Yee3 chicken rice": "Chicken rice, steamed",
"Yuen Kee pork and chive dumplings, boiled": "Pan-fried pork dumplings",
"Lanzhou beef lamian": "Pho",
"Samy & Min bak kut teh": "Bak kut teh with rice",
"Dirty duck with rice": "Roast duck rice",
"De.Wan nasi lemak with rendang": "Nasi lemak biasa",
"De.Wan ikan bakar set": "Ikan bakar with rice and ulam",
"Malaya Heritage nasi lemak": "Nasi lemak biasa",
"Serai nasi lemak with ayam berempah": "Nasi lemak biasa",
"Serai mee goreng": "Mee goreng mamak",
"Grandmama's nyonya fried rice": "Nasi goreng kampung",
"Grandmama's asam laksa": "Asam laksa",
"Celadon green curry chicken": "Green curry chicken with rice",
"Celadon tom yum goong": "Tom yum soup with rice",
"ParaThai pad kra pao chicken": "Pad kra pao with rice and egg",
"ParaThai mango sticky rice": "Mango sticky rice",
"Al Halabi mixed grill": "Satay ayam",
"Al-Amar chicken shawarma plate": "Bagel with cream cheese",
"Michelangelo's spaghetti carbonara": "Spaghetti carbonara",
"Michelangelo's margherita pizza": "Pizza",
"13DE Marzo pizza": "Pizza",
"La Boca beef tacos": "Bagel with cream cheese",
"KGB classic beef burger": "Cheeseburger",
"Weissbrau pork knuckle": "Steak with vegetables",
"Arthur's fish and chips": "Fish and chips",
"Two Sons mussels in white wine": "Butter prawns",
"O'Galito Portuguese grilled chicken": "Grilled chicken chop with salad",
"Ocean Twe12ve seafood pasta": "Spaghetti carbonara",
"Food Republic food court meal": "Economy rice: 1 meat, 2 veg",
"Food Place food court meal": "Economy rice: 1 meat, 2 veg",
"Da On galbi set": "Korean BBQ, typical meal",
"Tomacado salad bowl": "Chicken caesar salad",
"LAC protein bowl": "Economy rice: 1 meat, 2 veg",
"Okonomi by Tokyo Don okonomiyaki": "Takoyaki",
"Okonomi by Tokyo Don salmon don": "Salmon sashimi and rice",
"Cucur udang": "Pisang goreng",
"A Decade nasi lemak ayam goreng": "Nasi lemak biasa",
"A Decade mee goreng mamak": "Mee goreng mamak",
"A Decade roti bakar with kaya": "Kaya toast set",
"ABC One chicken chop rice": "Hainanese chicken chop",
"ABC One fried rice": "Nasi goreng kampung",
"Bone soup rice noodles": "Pho",
"Spicy bone soup rice noodles": "Pho",
"Auntea Jenny Hong Kong lemon tea": "Bubble milk tea",
"Auntea Jenny milk tea with boba": "Bubble milk tea",
"Auntea Jenny mango pomelo sago": "Tau fu fah, less syrup",
"Ah Cheng Penang assam laksa": "Asam laksa",
"Ah Cheng curry laksa": "Curry laksa",
"Ah Cheng Ipoh hor fun": "Ipoh hor fun",
"An Viet pho bo": "Pho",
"An Viet bun cha": "Pho",
"An Viet goi cuon (fresh spring rolls)": "Pisang goreng",
"An Viet Vietnamese iced coffee": "Latte, iced",
"Andaman grilled seafood platter, per person": "Fish and chips",
"Andaman tom yum seafood soup": "Tom yum soup with rice",
"Ayam penyet with rice": "Nasi ayam penyet",
"Ayam bakar with rice": "Ikan bakar with rice and ulam",
"Lele penyet with rice": "Nasi ayam penyet",
"Bebek goreng with rice": "Nasi ayam penyet",
"Baskin Robbins single scoop, cup": "Ice cream",
"Baskin Robbins double scoop in a waffle cone": "Waffle with ice cream",
"Baskin Robbins ice cream cake": "Chocolate cake",
"Genki Sushi salmon nigiri": "Sushi plates",
"Genki Sushi typical meal": "Sushi plates",
"Genki Sushi salmon aburi with mayo": "Sushi plates",
"Sushi Jiro typical meal": "Sushi plates",
"Nanyang Cafe kaya butter toast set": "Kaya toast set",
"Nanyang Cafe curry mee": "Curry laksa",
"Nanyang Cafe dry chili pan mee": "Pan mee, soup",
"Taiwan spicy beef noodles": "Boat noodles",
"Taiwan braised pork rice": "Sweet and sour pork with rice",
"MET oolong milk tea": "Bubble milk tea",
"MET fruit tea": "Bubble milk tea",
"MET matcha latte": "Matcha latte, iced",
"Beutea brewed milk tea": "Bubble milk tea",
"Beutea cheese foam tea": "Bubble milk tea",
"Itsu salmon teriyaki bento": "Chicken teriyaki don",
"Itsu chicken katsu don": "Tonkatsu set",
"Kinori salmon nori taco": "Sushi plates",
"Kinori spicy tuna nori taco": "Sushi plates",
"Jordan HK baked pork chop rice": "Baked pork chop rice",
"Jordan HK satay beef noodles": "Boat noodles",
"Jordan HK French toast": "French toast",
"Jordan HK milk tea, iced": "Bubble milk tea",
"Gordon Ramsay beef Wellington": "Steak with vegetables",
"Gordon Ramsay sticky toffee pudding": "Chocolate cake",
"Gordon Ramsay Street Pizza margherita slice": "Pizza",
"Gordon Ramsay Street Pizza pepperoni slice": "Pizza",
"Thai Thai tom yum goong": "Tom yum soup with rice",
"Thai Thai pineapple fried rice": "Nasi goreng kampung",
"Thai Thai green curry chicken": "Green curry chicken with rice",
"Rakuzen bento set": "Chicken teriyaki don",
"Rakuzen chirashi don": "Salmon sashimi and rice",
"Din Tai Fung xiao long bao": "Xiao long bao",
"Din Tai Fung chicken xiao long bao": "Xiao long bao",
"Din Tai Fung egg fried rice with shrimp": "Nasi goreng kampung",
"Din Tai Fung braised beef noodle soup": "Minced pork soup",
"Din Tai Fung spicy wontons": "Fried wantan",
"Din Tai Fung stir-fried string beans": "Stir-fried greens",
"Dandy grilled chicken plate": "Hainanese chicken chop",
"Dandy beef burger": "Cheeseburger",
"Simple Life brown rice set": "Economy rice: 1 meat, 2 veg",
"Simple Life vegetarian laksa": "Curry laksa",
"Black Tap All-American burger": "Cheeseburger",
"Black Tap CrazyShake, cookies and cream": "Chocolate milkshake",
"Black Tap buffalo wings": "Korean fried chicken, soy garlic",
"Pokok big breakfast": "Big breakfast",
"Pokok iced latte": "Latte, iced",
"Pokok truffle fries": "French fries",
"Mantra chicken biryani": "Nasi briyani ayam",
"Mantra butter chicken with naan": "Butter chicken with naan",
"Mantra masala dosa": "Thosai with dhal and chutney",
"Verrona Hills croissant": "Butter croissant",
"Verrona Hills basque cheesecake": "Cheesecake",
"The Coach iced latte": "Latte, iced",
"Taiwanese popcorn chicken": "Chicken nuggets",
"Taiwanese XXL fried chicken": "Korean fried chicken, soy garlic",
"Walking Hotpot, per person": "Hot pot, typical meal",
"Johor mini steamboat, per person": "Hot pot, typical meal",
"Mr Bean soya milk": "Mango lassi",
"Mr Bean soya pancake with peanut": "Waffle with ice cream",
"Hikiniku To Come hamburg steak set": "Steak with vegetables",
"Le Bourne hot chocolate": "Chocolate bar",
"Le Bourne chocolate pralines": "Chocolate bar",
"DubuYo sundubu jjigae": "Army stew (budae jjigae)",
"DubuYo bibimbap": "Bibimbap",
"DubuYo topokki": "Tteokbokki",
"Koong Woh Tong herbal jelly": "Tau fu fah, less syrup",
"Koong Woh Tong herbal tea": "Soft drink",
"Chez Choux cream puff": "Beard Papa original cream puff",
"Beryl's chocolate lava cake": "Chocolate cake",
"Beryl's hot chocolate": "Chocolate bar",
"Miga beef bulgogi set": "Bibimbap",
"Wok Thai pad kra pao chicken": "Pad kra pao with rice and egg",
"Wok Thai pad see ew": "Char kway teow",
"Torii teppanyaki chicken set": "Chicken teriyaki don",
"BreadStory sausage bun": "Char siu bao",
"BreadStory cream bun": "Char siu bao",
"Ramen Matsuri chicken shoyu ramen": "Ramen, tonkotsu",
"Ramen Matsuri spicy miso ramen": "Ramen, tonkotsu",
"Santan nasi lemak": "Nasi lemak biasa",
"Santan Pak Nasser's nasi lemak": "Nasi lemak biasa",
"Chizu cheese foam tea": "Bubble milk tea",
"Village roast duck rice": "Roast duck rice",
"Village roast duck, quarter": "Roast duck rice",
"Aftermeal waffle with ice cream": "Waffle with ice cream",
"Whopper Jr": "Cheeseburger",
"Double Whopper with cheese": "Cheeseburger",
"Mushroom Swiss beef burger": "Cheeseburger",
"Tendercrisp chicken burger": "Cheeseburger",
"Burger King chicken royale": "Cheeseburger",
"Burger King fish burger": "Cheeseburger",
"Burger King onion rings": "Cheeseburger",
"Burger King fries": "Cheeseburger",
"Burger King nuggets": "Chicken nuggets",
"Burger King sundae, chocolate": "Cheeseburger",
"6-inch Subway Melt": "Bagel with cream cheese",
"6-inch Italian B.M.T. (chicken ham, pepperoni, salami)": "Bagel with cream cheese",
"6-inch Chicken Slice sub": "Bagel with cream cheese",
"6-inch Steak & Cheese sub": "Steak with vegetables",
"6-inch Spicy Italian sub": "Bagel with cream cheese",
"6-inch egg mayo sub": "Bagel with cream cheese",
"6-inch seafood sensation sub": "Bagel with cream cheese",
"Subway salad, roasted chicken": "Chicken caesar salad",
"Subway chocolate chip cookie": "Chocolate chip cookie",
"Nando's half chicken": "Grilled chicken chop with salad",
"Nando's quarter chicken breast with 2 regular sides": "Grilled chicken chop with salad",
"Nando's chicken butterfly": "Grilled chicken chop with salad",
"Nando's chicken wrap": "Bagel with cream cheese",
"Nando's chicken burger": "Cheeseburger",
"Nando's peri-peri chips": "French fries",
"Nando's spicy rice": "Stir-fried greens",
"Nando's macho peas": "Stir-fried greens",
"Nando's corn on the cob": "Stir-fried greens",
"Texas Chicken spicy chicken, 2 pieces": "KFC Original Recipe, thigh",
"Texas Chicken tenders": "Chicken nuggets",
"Texas Chicken sandwich": "Cheeseburger",
"Texas Chicken honey butter rice bowl": "Korean fried chicken, soy garlic",
"Texas Chicken mashed potato": "Stir-fried greens",
"Texas Chicken cajun fries": "French fries",
"4Fingers wings, soy garlic": "Korean fried chicken, soy garlic",
"4Fingers wings, hot and spicy": "Korean fried chicken, soy garlic",
"4Fingers drumsticks, soy garlic": "Korean fried chicken, soy garlic",
"4Fingers crispy chicken burger": "Cheeseburger",
"4Fingers seaweed fries": "French fries",
"A&W Mozza burger": "Cheeseburger",
"A&W Teen burger": "Cheeseburger",
"A&W chicken, 2 pieces": "KFC Original Recipe, thigh",
"A&W waffle with ice cream": "Waffle with ice cream",
"A&W curly fries": "French fries",
"A&W root beer": "Soft drink",
"Auntie Anne's pretzel, almond": "Pretzel, original",
"Auntie Anne's pretzel, sour cream and onion": "Pretzel, original",
"Auntie Anne's pretzel dogs": "Pretzel, original",
"Auntie Anne's pretzel bites, original": "Pretzel, original",
"Auntie Anne's lemonade": "Soft drink",
"Secret Recipe Marble Cheese cake": "Cheesecake",
"Secret Recipe Chocolate Indulgence": "Chocolate cake",
"Secret Recipe Durian cake": "Chocolate cake",
"Secret Recipe tom yam fried rice": "Nasi goreng kampung",
"Secret Recipe chicken lasagna": "Spaghetti carbonara",
"Secret Recipe nasi lemak with chicken rendang": "Nasi lemak biasa",
"Secret Recipe fish and chips": "Fish and chips",
"PappaRich nasi lemak with curry chicken": "Nasi lemak biasa",
"PappaRich curry laksa": "Curry laksa",
"PappaRich wantan mee": "Wantan mee, dry",
"PappaRich roti canai with curry": "Roti canai kosong",
"PappaRich chicken chop rice": "Hainanese chicken chop",
"PappaRich char kway teow": "Char kway teow",
"PappaRich white coffee, iced": "Latte, iced",
"Kenny Rogers quarter chicken with 2 sides": "Grilled chicken chop with salad",
"Kenny Rogers half chicken with 3 sides": "Grilled chicken chop with salad",
"Kenny Rogers muffin": "Scone with jam and cream",
"Kenny Rogers mac and cheese": "Spaghetti carbonara",
"Boat Noodle beef noodles": "Boat noodles",
"Boat Noodle tom yam noodles": "Tom yum soup with rice",
"Boat Noodle pad thai": "Pad thai",
"Boat Noodle Thai milk tea": "Bubble milk tea",
"Boat Noodle mango sticky rice": "Mango sticky rice",
"Longjing xiao long bao": "Xiao long bao",
"Longjing Dongpo pork rice": "Sweet and sour pork with rice",
"Longjing shrimp fried rice": "Nasi goreng kampung",
"Chakri massaman chicken curry": "Green curry chicken with rice",
"Chakri basil minced pork rice": "Pad kra pao with rice and egg",
"Cili Kampung ikan keli berlado": "Nasi ayam penyet",
"Absolute Thai pad thai": "Pad thai",
"Absolute Thai green curry chicken": "Green curry chicken with rice",
"Absolute Thai tom yum seafood": "Tom yum soup with rice",
"Absolute Thai basil chicken rice": "Chicken rice, steamed",
"Chica Bonita chicken tacos": "Bagel with cream cheese",
"Chica Bonita beef burrito": "Bagel with cream cheese",
"Chica Bonita nachos with cheese": "French fries",
"Robataya grilled skewers, per person": "Satay ayam",
"Robataya grilled saba set": "Steamed fish with rice and greens",
"Matcha Eight matcha latte": "Matcha latte, iced",
"Matcha Eight matcha cake": "Matcha latte, iced",
"Teochew braised duck, per person": "Roast duck rice",
"Teochew steamed pomfret, per person": "Steamed fish with rice and greens",
"Teochew porridge with sides": "Chicken porridge",
"Elementary fish and chips": "Fish and chips",
"Elementary beef pie": "Apple pie",
"Las Meninas seafood paella, per person": "Nasi goreng kampung",
"Quivo Mediterranean grilled chicken": "Grilled chicken chop with salad",
"a'rest iced latte": "Latte, iced",
"Reborn iced latte": "Latte, iced",
"36 Degree jalapeno bagel": "Bagel with cream cheese",
"All About Chew chocolate chip cookie": "Chocolate chip cookie",
"AiFOKATO affogato": "Ice cream",
"Ticco milk tea": "Bubble milk tea",
"Nak Nak boneless thigh, soy garlic": "Korean fried chicken, soy garlic",
"Nak Nak boneless thigh, gochujang": "Korean fried chicken, soy garlic",
"Nak Nak beef burger": "Cheeseburger",
"Nak Nak loaded fries": "French fries",
"Fong Lye three cup chicken rice": "Chicken rice, steamed",
"Fong Lye beef noodle soup": "Minced pork soup",
"Rooster& chicken rice": "Chicken rice, steamed",
"Meriah platter": "Latte, iced",
"Meriah banana fritters": "Pisang goreng",
"Meriah chilled mango sago": "Tau fu fah, less syrup",
"DayOne katsu curry pork chop omelette rice": "Chicken katsu curry rice",
"DayOne salmon don": "Salmon sashimi and rice",
"DayOne teriyaki chicken rice": "Chicken rice, steamed",
"Kanbe kuro ramen": "Ramen, tonkotsu"
}
SHOP_EXTRA = {
"korean bbq samgyeopsal grill": "Korean BBQ, typical meal",
"yakiniku grill table": "Korean BBQ, typical meal",
"xiao long bao": "Xiao long bao",
"beef wellington": "Steak with vegetables",
"pad kra pao": "Pad kra pao with rice and egg",
"mango sticky rice": "Mango sticky rice",
"fried chicken wings korean": "Korean fried chicken, soy garlic",
"hong kong pineapple bun": "Pineapple bun with butter",
"baked pork chop rice hong kong": "Baked pork chop rice",
"unagi don": "Unagi don",
"pan fried dumplings": "Pan-fried pork dumplings",
"croissant": "Butter croissant",
"tiramisu": "Tiramisu",
"iced latte": "Latte, iced",
"big breakfast plate": "Big breakfast",
"ipoh hor fun": "Ipoh hor fun"
}
SHOP_PER_DISH = 30
by_look = {}
for q, look in list(SHOP_LOOKS.items()) + list(SHOP_EXTRA.items()):
    by_look.setdefault(look, []).append(q)
if WEB_PHOTOS:
    for look, qs in by_look.items():
        lab = "sh_" + re.sub(r"[^a-z0-9]+", "_", look.lower()).strip("_"); d = f"/tmp/shops/{lab}"; os.makedirs(d, exist_ok=True)
        jobs = []
        for qi, q in enumerate(qs):
            jobs += [(u, f"{d}/{qi}_{j}.jpg") for j, u in enumerate(web_photos(q, SHOP_PER_DISH))]; time.sleep(1.5)
        with ThreadPoolExecutor(6) as pool: files = [f for f in pool.map(grab, jobs) if f]
        if len(files) >= 25: MAP_CM[lab] = look; CM_FILES[lab] = files
        print(f"shops, {look}: {len(files)} photos from {len(qs)} dishes" + ("" if len(files) >= 25 else "  (too few, left out)"))

CLASSES = sorted(set(MAP_MY.values()) | set(MAP_F101.values()) | set(MAP_IN.values()) | set(MAP_CM.values()))   # one class per dish in the app, so the same dish from two sources is one class
IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes")

class Files(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        fn, y = self.rows[k]
        return self.tf(Image.open(fn).convert("RGB")), y

cm_train, cm_val = [], []
for lab, files in CM_FILES.items():
    random.Random(SEED).shuffle(files); nv = min(80, max(5, len(files) // 5))
    y = IDX[MAP_CM[lab]]; cm_val += [(fn, y) for fn in files[:nv]]; cm_train += [(fn, y) for fn in files[nv:nv + PER_CLASS]]   # capped so big groups don't drown the rest
print(len(cm_train), "extra training photos from Wikimedia Commons")


def pick(ds, names, mapping, cap, seed=SEED):
    """Row indices for the wanted classes, at most `cap` per class."""
    labels = ds["label"]                                # reads the label column only, no image decoding
    by = {}
    for i, l in enumerate(labels):
        n = names[l]
        if n in mapping: by.setdefault(n, []).append(i)
    rng = random.Random(seed); out = []
    for n, idx in by.items():
        rng.shuffle(idx); out += [(i, IDX[mapping[n]]) for i in idx[:cap]]
    return out

class Photos(Dataset):
    def __init__(self, ds, rows, tf): self.ds, self.rows, self.tf = ds, rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        for step in range(20):   # a few photos in the public datasets are broken files: skip to the next one
            i, y = self.rows[(k + step) % len(self.rows)]
            try: return self.tf(self.ds[i]["image"].convert("RGB")), y
            except Exception: continue
        raise RuntimeError("too many unreadable photos in a row")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def square(img):
    """Centre square crop then resize to 224. The app prepares photos exactly this way."""
    w, h = img.size; m = min(w, h)
    return img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2)).resize((224, 224), Image.BILINEAR)

train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.55, 1.0)), T.RandomHorizontalFlip(),
                      T.ColorJitter(0.3, 0.3, 0.3, 0.05), T.ToTensor(), T.Normalize(MEAN, STD)])
test_tf = T.Compose([T.Lambda(square), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = ConcatDataset([Photos(my["train"], pick(my["train"], my_names, MAP_MY, PER_CLASS), train_tf),
                          Photos(f101["train"], pick(f101["train"], f_names, MAP_F101, PER_CLASS), train_tf)]
                         + ([Photos(india["train"], pick(india["train"], in_names, MAP_IN, PER_CLASS), train_tf)] if india else [])
                         + ([Files(cm_train, train_tf)] if cm_train else []))
val_ds = ConcatDataset([Photos(my["test"], pick(my["test"], my_names, MAP_MY, 150), test_tf),
                        Photos(f101["validation"], pick(f101["validation"], f_names, MAP_F101, 150), test_tf)]
                       + ([Files(cm_val, test_tf)] if cm_val else []))
print(len(train_ds), "training photos,", len(val_ds), "test photos")

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)

## Train

First only the new final layer is trained, then the whole network at a lower learning rate.

In [ ]:
model = timm.create_model("mobilenetv3_large_100", pretrained=True, num_classes=len(CLASSES)).to(DEVICE)
loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")

def evaluate():
    model.eval(); right = np.zeros(len(CLASSES)); seen = np.zeros(len(CLASSES))
    with torch.no_grad():
        for x, y in val_dl:
            p = model(x.to(DEVICE)).argmax(1).cpu()
            for a, b in zip(p.tolist(), y.tolist()): seen[b] += 1; right[b] += a == b
    return right.sum() / seen.sum(), right / np.maximum(seen, 1)

def run(epochs, lr, head_only):
    for n, p in model.named_parameters(): p.requires_grad = (not head_only) or n.startswith("classifier")
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    for ep in range(epochs):
        model.train(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                loss = loss_fn(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); total += loss.item()
        acc, _ = evaluate()
        print(f"epoch {ep + 1}/{epochs}  loss {total / len(train_dl):.3f}  test accuracy {acc:.1%}")

run(EPOCHS_HEAD, 2e-3, head_only=True)
run(EPOCHS_ALL, 4e-4, head_only=False)

acc, per = evaluate()
print(f"\nFinal test accuracy: {acc:.1%}\n")
for c, a in sorted(zip(CLASSES, per), key=lambda t: t[1]): print(f"{a:6.1%}  {c}")

## Save the two files for the app

In [ ]:
import onnx, onnxruntime as ort

model.eval().cpu()
dummy = torch.zeros(1, 3, 224, 224)
kw = dict(input_names=["input"], output_names=["logits"], opset_version=17)
try: torch.onnx.export(model, dummy, "food.onnx", dynamo=False, **kw)
except TypeError: torch.onnx.export(model, dummy, "food.onnx", **kw)
onnx.checker.check_model(onnx.load("food.onnx"))

import re
json.dump([{"label": re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_"), "food": c} for c in CLASSES], open("labels.json", "w"), indent=1)

# check the saved model gives the same answers as the trained one
sess = ort.InferenceSession("food.onnx")
x, y = next(iter(val_dl))
a = sess.run(None, {"input": x[:1].numpy()})[0]
b = model(x[:1]).detach().numpy()
print("largest difference between saved and trained model:", float(np.abs(a - b).max()))

import os
print(f"food.onnx: {os.path.getsize('food.onnx') / 1e6:.1f} MB.  labels.json: {len(CLASSES)} classes.")
print("Download both files from the Output panel and put them in the app's model folder.")

## Adding your own dishes later

To teach the model a dish that is not in the list:

1. Collect at least 80 to 100 of your own photos of it, taken the way you normally would.
2. Add them as another dataset class, add a line to `MAP` with the exact dish name from the app's food list, and run the notebook again.

Only use photos you took yourself or that are published under a licence that allows reuse.